<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/form-nport/extract-etf-constituents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This Python tutorial shows you how to extract the list of constituents for a given ETF ticker from its most recent N-PORT filing. The process works for any ETF, such as S&P 500, Russell 3000 and many more.

Process overview:

- Find the accession number of the most recently filed N-PORT filing for a given ETF ticker. Use the Query API to search the EDGAR database. 
- Load the N-PORT filing for the given accession number into a pandas DataFrame using the N-PORT API.
- Extract constituents of ETF from DataFrame

The example uses the iShares Core S&P 500 ETF with the ticker symbol IVV.

In [ ]:
!pip install sec-api

In [65]:
from sec_api import QueryApi

api_key = 'YOUR_API_KEY'
queryApi = QueryApi(api_key=api_key)

# iShares Core S&P 500 ETF
etf_ticker = 'IVV'
# iShares Russell 3000 ETF
# etf_ticker = 'IWV'
# Vanguard High Dividend Yield ETF
# etf_ticker = 'VYM'


query = {
    "query": "formType:\"NPORT-P\" AND seriesAndClassesContractsInformation.classesContracts.ticker:" + etf_ticker,
    "from": "0",
    "size": "10",
    "sort": [{"filedAt": {"order": "desc"}}]
}

response = queryApi.get_filings(query)

In [66]:
import pandas as pd

metadata = pd.DataFrame.from_records(response['filings'])
metadata.head(3)

,id,accessionNo,cik,ticker,companyName,companyNameLong,formType,description,filedAt,linkToTxt,linkToHtml,linkToXbrl,linkToFilingDetails,entities,documentFormatFiles,dataFiles,seriesAndClassesContractsInformation,periodOfReport
0,1c03495da372416a860eaa0b12fbe71c,0001752724-23-039243,1100663,,iSHARES TRUST,iSHARES TRUST (Filer),NPORT-P,Form NPORT-P - Monthly Portfolio Investments R...,2023-02-24T13:00:15-05:00,https://www.sec.gov/Archives/edgar/data/110066...,https://www.sec.gov/Archives/edgar/data/110066...,,https://www.sec.gov/Archives/edgar/data/110066...,"[{'companyName': 'iSHARES TRUST (Filer)', 'cik...","[{'sequence': '1', 'documentUrl': 'https://www...",[],"[{'series': 'S000004310', 'name': 'iShares Cor...",2022-12-31
1,a1f94bcbf640ed6808e2f4785f87289f,0001752724-22-268673,1100663,,iSHARES TRUST,iSHARES TRUST (Filer),NPORT-P,Form NPORT-P - Monthly Portfolio Investments R...,2022-11-28T10:15:28-05:00,https://www.sec.gov/Archives/edgar/data/110066...,https://www.sec.gov/Archives/edgar/data/110066...,,https://www.sec.gov/Archives/edgar/data/110066...,"[{'companyName': 'iSHARES TRUST (Filer)', 'cik...","[{'sequence': '1', 'documentUrl': 'https://www...",[],"[{'series': 'S000004310', 'name': 'iShares Cor...",2022-09-30
2,0321c565e9c40c3ed74b03d633110e42,0001752724-22-193652,1100663,,iSHARES TRUST,iSHARES TRUST (Filer),NPORT-P,Form NPORT-P - Monthly Portfolio Investments R...,2022-08-25T14:39:08-04:00,https://www.sec.gov/Archives/edgar/data/110066...,https://www.sec.gov/Archives/edgar/data/110066...,,https://www.sec.gov/Archives/edgar/data/110066...,"[{'companyName': 'iSHARES TRUST (Filer)', 'cik...","[{'sequence': '1', 'documentUrl': 'https://www...",[],"[{'series': 'S000004310', 'name': 'iShares Cor...",2022-06-30


Let's get the accession number of the most recently filed N-PORT filing for our iShares Core S&P 500 ETF.

In [67]:
accession_number = metadata.iloc[0]['accessionNo']
print(accession_number)

0001752724-23-039243


Next, we load the content of the N-PORT filing into a pandas DataFrame. The first step is to retrieve the filing as JSON formatted object using the `nportApi.get_data(search_query)` function. The `search_query` tells the N-PORT search engine to return a single match (`"size": "1"`) where the match has to include our previously identified accession number (`"query": "accessionNo:\"{accession_number}\"".format(accession_number=accession_number)`).

We convert the JSON formatted filing into a DataFrame using `pd.json_normalize(response["filings"][0])`. Extracting the holdings (or constituents) disclosed in the filing and generating a new DataFrame with the data is accomplished by using `pd.json_normalize(filing['invstOrSecs'].iloc[0])`.

In [68]:
import pandas as pd
from sec_api import FormNportApi

nportApi = FormNportApi(api_key=api_key)

search_query = {
    "query": "accessionNo:\"{accession_number}\"".format(accession_number=accession_number),
    "from": "0",
    "size": "1",
    "sort": [{"filedAt": {"order": "desc"}}],
}

response = nportApi.get_data(search_query)

filing = pd.json_normalize(response["filings"][0])

holdings = pd.json_normalize(filing['invstOrSecs'].iloc[0])

filing.drop(labels=['invstOrSecs'], axis=1, inplace=True)

In [69]:
fund_name = filing['genInfo.seriesName'].iloc[0]
print('Fund name:', fund_name)

Fund name: iShares Core S&P 500 ETF


In [70]:
from google.colab.data_table import DataTable
DataTable.max_columns = 40

print('Constituents of {fund_name} ({ticker})'.format(fund_name=fund_name, ticker=etf_ticker))
print('-------------------------------------------------')
holdings

Constituents of iShares Core S&P 500 ETF (IVV)
-------------------------------------------------


,name,lei,title,cusip,balance,units,curCd,valUSD,pctVal,payoffProfile,...,issuerConditional.issuerCat,derivativeInfo.futrDeriv.counterparties,derivativeInfo.futrDeriv.payOffProf,derivativeInfo.futrDeriv.descRefInstrmnt.indexBasketInfo.indexName,derivativeInfo.futrDeriv.descRefInstrmnt.indexBasketInfo.indexIdentifier,derivativeInfo.futrDeriv.expDate,derivativeInfo.futrDeriv.notionalAmt,derivativeInfo.futrDeriv.curCd,derivativeInfo.futrDeriv.unrealizedAppr,derivativeInfo.futrDeriv.derivCat
0,"CBRE Group, Inc.",52990016II9MJ2OSWA10,"CBRE Group, Inc., Class A",12504L109,2837947.0,NS,USD,2.184084e+08,0.075452,Long,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Brown & Brown, Inc.",549300PC8KTJ71XKFY89,"Brown & Brown, Inc.",115236101,2111367.0,NS,USD,1.202846e+08,0.041554,Long,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Host Hotels & Resorts, Inc.",N6EL63S0K3PB1YFTDI24,"Host Hotels & Resorts, Inc.",44107P104,6422159.0,NS,USD,1.030757e+08,0.035609,Long,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Intercontinental Exchange, Inc.",5493000F4ZO33MV32P92,"Intercontinental Exchange, Inc.",45866F104,5016736.0,NS,USD,5.146669e+08,0.177798,Long,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Oracle Corp.,1Z4GXXU7ZHVWFCD8TV52,Oracle Corp.,68389X105,13802817.0,NS,USD,1.128242e+09,0.389766,Long,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,"Equinix, Inc.",549300EVUN2BTLJ3GT74,"Equinix, Inc.",29444U700,831133.0,NS,USD,5.444170e+08,0.188076,Long,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
502,"eBay, Inc.",OML71K8X303XQONU6T67,"eBay, Inc.",278642103,4874229.0,NS,USD,2.021343e+08,0.069830,Long,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
503,AO Smith Corp.,549300XG4US7UJNECY36,AO Smith Corp.,831865209,1139495.0,NS,USD,6.522469e+07,0.022533,Long,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
504,Cintas Corp.,N/A,Cintas Corp.,172908105,775287.0,NS,USD,3.501351e+08,0.120959,Long,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The next example extracts the list of borrowers from the N-PORT filing and converts it into a new DataFrame `borrowers`.

In [71]:
borrowers = pd.json_normalize(filing['fundInfo.borrowers.borrower'].iloc[0])

filing.drop(labels=['fundInfo.borrowers.borrower'], axis=1, inplace=True)

borrowers

,aggrVal,lei,name
0,4.446460e+05,RUC0QBLBRPRCU4W1NE59,BMO Capital Markets
1,4.622688e+07,RCNB6OTYUAMMP879YW96,BNP PARIBAS SECURITIES CORP
2,6.158053e+07,BFM8T61CT2L1QCEMIK50,UBS AG
3,2.735210e+08,MP6I5ZYZBEU3UXPYFY54,HSBC BANK PLC
4,4.276696e+07,T6FIZBDPKLYJKFCRVK44,UBS SECURITIES LLC
5,1.411006e+08,G5GSEF7VJP5I7OUK5573,BARCLAYS BANK PLC
6,1.707703e+07,58PU97L1C0WSRCWADL48,Jefferies LLC
7,3.005683e+08,549300BLWPABP1VNME36,Scotia Capital (USA) Inc
8,3.501195e+08,9R7GPTSO7KV3UQJZQ078,MORGAN STANLEY & CO. LLC (US EQUITY SEC LENDING)
9,3.318326e+06,549300L8G1E7ZHVEOG75,Natixis Securities Americas LLC


Now let us inspect the N-PORT filing and all of its values.

In [72]:
print('{fund_name} ({ticker}) N-PORT Filing Overview'.format(fund_name=fund_name, ticker=etf_ticker))
print('-------------------------------------------------')

filing_stacked = filing.stack()[0]

for index, value in filing_stacked.items():
  if isinstance(value, list):
    continue
  key = index if len(index) < 50 else '...' + index[len(index) - 47:]
  padding_length = 70 - len(str(key)+str(value));
  print(key, ''.ljust(padding_length, ' '), value)

iShares Core S&P 500 ETF (IVV) N-PORT Filing Overview
-------------------------------------------------
submissionType                                                   NPORT-P
accessionNo                                         0001752724-23-039243
filedAt                                        2023-02-24T13:00:15-05:00
id                                      88bfb8328b0538f0713310f01f5ff8e9
filerInfo.filer.issuerCredentials.cik                         0001100663
filerInfo.filer.issuerCredentials.ccc                           XXXXXXXX
filerInfo.seriesClassInfo.seriesId                            S000004310
genInfo.regName                                            iShares Trust
genInfo.regFileNumber                                          811-09729
genInfo.regCik                                                0001100663
genInfo.regLei                                      5493000860OXIC4B5K91
genInfo.regStreet1                                     400 Howard Street
genInfo.regCity     